# Convergent Scripture, Divergent Rulings — reproducible analysis

This notebook reproduces the headline results of the paper from the **cached data**
in `data/out/` — no GPU, no model download, no network access. It:

1. loads the 105-verse × 5-edition corpus,
2. recomputes the cross-edition **similarity matrix** from the LaBSE embeddings (and
   asserts it matches the committed values),
3. shows which **muʿāmalāt domains** diverge most,
4. re-runs the **Wilcoxon** and **Kruskal–Wallis** tests in Python,
5. plots the **two-layer decoupling** (translation similarity vs fatwa distance).

Method background: Lange, van Kuppevelt & van der Zwaan (2021), *Text Mining Islamic Law*.

In [ ]:
import itertools
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import wilcoxon, kruskal, rankdata

ROOT = Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists():
    ROOT = ROOT.parent
DATA, OUT = ROOT/"data", ROOT/"data"/"out"
LANGS = ["id","ms","th","tl","bn"]
NAMES = {"id":"Indonesia","ms":"Malaysia","th":"Thailand","tl":"Philippines","bn":"Brunei"}
print("repo root:", ROOT)

## 1. The corpus

105 muʿāmalāt anchor verses, each rendered by five national editions, joined on the
sūra:āya key. See `DATA_SOURCES.md` for provenance.

In [ ]:
corpus = pd.read_csv(DATA/"economic_verses_aligned.csv")
print(corpus.shape[0], "verses;", "editions:", [c for c in corpus.columns if c.startswith("terjemah_")])
corpus.loc[corpus.surah_ayat=="83:1", ["surah_ayat","terjemah_id","terjemah_ms","terjemah_bn","terjemah_th","terjemah_tl"]].T

## 2. Translation layer: cross-edition similarity

Each rendering is a LaBSE vector. For every verse we take the cosine between each
pair of editions, then average across the 105 verses. Dark = more similar.

In [ ]:
X = np.load(OUT/"embeddings.npy")          # 525 x 768, order [id,ms,th,tl,bn] x 105
n = 105
blocks = {lg: X[i*n:(i+1)*n] for i, lg in enumerate(LANGS)}
M = pd.DataFrame(1.0, index=LANGS, columns=LANGS)
for a, b in itertools.combinations(LANGS, 2):
    c = float((blocks[a]*blocks[b]).sum(axis=1).mean())   # unit vectors -> dot = cosine
    M.loc[a, b] = M.loc[b, a] = round(c, 4)

ref = pd.read_csv(OUT/"lang_similarity_matrix.csv", index_col=0)
assert np.allclose(M.values, ref.values, atol=5e-3), "recomputed != committed matrix"
print("OK: recomputed matrix matches the committed lang_similarity_matrix.csv")

Md = M.rename(index=NAMES, columns=NAMES)
plt.figure(figsize=(5.4,4.6))
sns.heatmap(Md, annot=True, fmt=".2f", cmap="mako_r", vmin=0.65, vmax=1.0,
            square=True, linewidths=1, cbar_kws={"label":"mean cosine"})
plt.title("Cross-edition similarity of muʿāmalāt renderings"); plt.tight_layout(); plt.show()
print("Closest pair:", Md.where(~np.eye(5,dtype=bool)).stack().idxmax(),
      "=", round(Md.where(~np.eye(5,dtype=bool)).stack().max(),3))

The three Malay-family editions (Indonesia, Malaysia, Brunei) cluster tightly, and
**Malaysia–Brunei is the closest pair of all (0.92)** — convergence tracks language family.

## 3. Which muʿāmalāt domains diverge most

`divergence = 1 − mean cross-edition cosine`, per domain (domains with ≥ 3 verses,
matching the Kruskal–Wallis test below).

In [ ]:
dom = pd.read_csv(OUT/"domain_divergence.csv")
dom = dom[dom.n_ayat >= 3].sort_values("divergence")
plt.figure(figsize=(7,5))
plt.barh(dom.domain, dom.divergence, color="#2c6b8f")
for y,(v,nn) in enumerate(zip(dom.divergence, dom.n_ayat)):
    plt.text(v+0.003, y, f"{v:.2f} (n={nn})", va="center", fontsize=8, color="grey")
plt.xlabel("divergence (1 − mean cross-edition cosine)")
plt.title("Cross-edition divergence by muʿāmalāt domain"); plt.tight_layout(); plt.show()

## 4. Statistical tests (reproduced in Python)

**(a)** Do the same-family (Malay) pairs agree more than cross-family pairs? Paired
Wilcoxon signed-rank, pairing by verse. **(b)** Does divergence differ by domain?
Kruskal–Wallis across domains with ≥ 3 verses.

In [ ]:
pw = pd.read_csv(OUT/"pairwise_long.csv")
wide = pw.pivot(index="surah_ayat", columns="pair", values="cosine")
malay = ["id-ms","id-bn","ms-bn"]
cross = [c for c in wide.columns if c not in malay]
within, crossm = wide[malay].mean(axis=1), wide[cross].mean(axis=1)

d = (within - crossm); d = d[d != 0]
V = rankdata(np.abs(d))[d.values > 0].sum()           # R-style V (sum of positive ranks)
_, p = wilcoxon(within, crossm, alternative="greater")
print(f"(a) Wilcoxon within>cross:  V = {V:.0f},  p = {p:.2e}   [paper: V=5564, p=3.05e-19]")

vd = pd.read_csv(OUT/"verse_divergence.csv")
big = vd.groupby("tema").filter(lambda g: len(g) >= 3)
groups = [g["divergence"].values for _, g in big.groupby("tema")]
H, pk = kruskal(*groups)
print(f"(b) Kruskal-Wallis by domain: H = {H:.1f}, df = {len(groups)-1}, p = {pk:.3f}   [paper: 35.9, 15, 0.002]")

## 5. The two layers decouple

For the three country-pairs that have **both** a translation edition and fatwa data,
plot translation similarity against fatwa distance. The most textually similar pair
(Malaysia–Brunei) sits at opposite ends of the ruling spectrum: similarity does not
predict the fatwa.

In [ ]:
fat = pd.read_csv(DATA/"fatwa_comparison.csv")
issues = ["Bank interest","Insurance/takaful","Crypto","Tawarruq"]
piv = fat.pivot_table(index="country", columns="issue", values="strictness_1to5")
lang = {"Indonesia":"id","Malaysia":"ms","Brunei":"bn"}
pairs = [("Indonesia","Malaysia"),("Indonesia","Brunei"),("Malaysia","Brunei")]

xs, ys, labs = [], [], []
for a, b in pairs:
    xs.append(float(M.loc[lang[a], lang[b]]))
    ys.append(float(np.mean([abs(piv.loc[a,i]-piv.loc[b,i]) for i in issues])))
    labs.append(f"{a}-{b}")

plt.figure(figsize=(7,5))
plt.scatter(xs, ys, s=220, c=["#4c78a8","#59a14f","#e15759"], edgecolor="white", zorder=3)
for x,y,l in zip(xs,ys,labs):
    plt.annotate(l,(x,y),xytext=(0,12),textcoords="offset points",ha="center",fontweight="bold")
plt.xlabel("Translation similarity (LaBSE mean cosine)")
plt.ylabel("Fatwa distance (mean strictness gap, 4 issues)")
plt.title("Two layers decouple: similar translations, divergent rulings")
plt.grid(alpha=.25); plt.tight_layout(); plt.show()
for l,x,y in zip(labs,xs,ys): print(f"{l:22} similarity={x:.3f}  fatwa-distance={y:.2f}")

---
**Takeaway.** Translation converges (governed by language family); fatwa diverges
(governed by economic-institutional position). The most similar pair of editions,
Malaysia–Brunei, anchors opposite ends of the fatwa spectrum.

To regenerate every figure and table (not just these), run `bash run_all.sh` from the
repo root. See `README.md` for the full pipeline.